In [85]:
import pandas as pd
import ast
df = pd.read_csv('datasets/11-7-2025-sp.csv')
gpa_map = {
    'A+': 4.0, 'A': 4.0, 'A-': 3.7,
    'B+': 3.3, 'B': 3.0, 'B-': 2.7,
    'C+': 2.3, 'C': 2.0, 'C-': 1.7,
    'D+': 1.3, 'D': 1.0, 'D-': 0.7,
    'F': 0.0, 'W': 0.0
}

In [86]:
def parse_grade_list(grades):
    """Convert CSV string to list of floats if needed."""
    if isinstance(grades, str):
        grades = ast.literal_eval(grades)
    return [float(x) for x in grades]

In [98]:
def avg_gpa_from_list(grades):
    """Calculate GPA from a grade list (last element = total students)."""
    grades = parse_grade_list(grades)
    total_students = grades[-1]
    grade_counts = grades[:-1]
    weighted_sum = sum(gpa * count for gpa, count in zip(gpa_map.values(), grade_counts))
    return weighted_sum / total_students if total_students > 0 else 0


In [97]:
def gpa_score(crn):
    """Return (professor + class GPA)  for a given CRN."""
    row = df[df['CRN'] == crn]
    if row.empty:
        return 0

    prof_grades = row["Mean Grade By Professor (A+..F,W,Students)"].values[0]
    class_grades = row["Mean Grade By Class (A+..F,W,Students)"].values[0]

    prof_gpa = avg_gpa_from_list(prof_grades)
    class_gpa = avg_gpa_from_list(class_grades)

    return (prof_gpa + class_gpa)

In [89]:
def percent_ge_from_list(grades, threshold):
    """Calculate percentage of students with GPA >= threshold."""
    grades = parse_grade_list(grades)
    total_students = grades[-1]
    grade_counts = grades[:-1]
    above_count = sum(count for gpa, count in zip(gpa_map.values(), grade_counts) if gpa >= threshold)
    return (above_count / total_students) * 100 if total_students > 0 else 0

In [90]:
def percent_ge_prof(crn, threshold):
    row = df[df['CRN'] == crn]
    if row.empty:
        return 0
    grades = row["Mean Grade By Professor (A+..F,W,Students)"].values[0]
    return percent_ge_from_list(grades, threshold)

In [91]:
def percent_ge_class(crn, threshold):
    row = df[df['CRN'] == crn]
    if row.empty:
        return 0
    grades = row["Mean Grade By Class (A+..F,W,Students)"].values[0]
    return percent_ge_from_list(grades, threshold)

In [92]:
def sum_ratings(crn):
    """Sum RMP, Excellent, Outstanding for a given CRN."""
    row = df[df['CRN'] == crn]
    if row.empty:
        return 0
    return (row['RMP'].values[0]) / 5

In [93]:
def total_class_score(crn, min_gpa1, min_gpa2):
    """Sum RMP sum + GPA score + %prof >= min_gpa1 + %class >= min_gpa2."""
    rmp_sum = sum_ratings(crn)
    gpa_val = gpa_score(crn)
    perc_prof = percent_ge_prof(crn, min_gpa1)
    perc_class = percent_ge_class(crn, min_gpa2)
    return rmp_sum + gpa_val + perc_prof + perc_class


In [95]:
# test case
print(gpa_score (30112))
print(total_class_score(30112, 3.7, 2.0))

6.010962596251968
131.66452494348903
